# Etapa 06 — MLflow e persistência dos modelos

Este notebook reúne a execução histórica real e o fluxo atual de tracking nativo.
A migração `mlflow_skops_v1` foi concluída;
Os próximos treinamentos usam `baseline_v2`, mantendo a mesma política científica.
[Runbook](../../docs/MLFLOW.md) e [arquitetura](../../docs/ARCHITECTURE.md).

Os resultados históricos abaixo foram extraídos dos outputs reais deste notebook
na revisão `6ba10b5` e apresentados como registros estáticos em Markdown.
As células executáveis atuais aguardam execução local; têm outputs vazios.


## Fluxo atual

Apenas quando houver motivo para um novo treinamento:

```bash
poetry run python -m fraud_detection_mlops.modeling.train run
```

O comando ajusta os três candidatos no treino, compara a validação, grava pipelines
skops e cria três runs principais no MLflow. Retorna `baseline_path`, `audit_path`
e as URIs nativas dos modelos. O teste permanece reservado.
Aplicar a refatoração exige os checks de qualidade; não exige repetir o baseline.

Para abrir os experimentos existentes e verificar seus artefatos:

```bash
make mlflow-ui
poetry run python -m fraud_detection_mlops.modeling.train verify "<baseline_path>"
poetry run python -m fraud_detection_mlops.modeling.tracking verify "<model_uri>"
```

O primeiro verify confere bytes e recalcula métricas sem carregar modelos.
O segundo confere formato, assinatura e carregamento no MLflow. A paridade com os
scores da validação é conferida na publicação de cada novo candidato.


## Conferir uma execução existente

As células seguintes verificam o baseline histórico explicitamente identificado.
Para uma execução nova, troque `ARTIFACT_VERSION`, `RUN_ID` e `MODEL_URI` pelos
valores retornados pelo treinamento. Não selecionamos a pasta mais recente.
Nenhuma destas células ajusta ou converte modelos.


In [1]:
from fraud_detection_mlops.bronze import DEFAULT_INVENTORY, load_inventory
from fraud_detection_mlops.modeling.train import DEFAULT_BASELINE, verify_baseline

ARTIFACT_VERSION = "baseline_v1"
RUN_ID = "f6d7aca720f74316b4183f97b6d866ab"
source_commit = load_inventory(DEFAULT_INVENTORY)["source"]["commit"]
run_dir = DEFAULT_BASELINE / source_commit / ARTIFACT_VERSION / RUN_ID
checked = verify_baseline(run_dir)
print(checked)

/home/wanderson/Documentos/fraud-detection-mlops/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'baseline_path': '/home/wanderson/Documentos/fraud-detection-mlops/data/processed/handbook/6e67dbd0a3bfe0d7ec33abc4bce5f37cd4ff0d6a/baseline_v1/f6d7aca720f74316b4183f97b6d866ab', 'run_id': 'f6d7aca720f74316b4183f97b6d866ab', 'verified_outputs': 10, 'selected_model': 'hist_gradient_boosting', 'comparison': {'dummy_prior': {'rows': 67255, 'fraud_count': 580, 'fraud_rate': 0.008623894134265111, 'average_precision': 0.008623894134265111, 'roc_auc': 0.5, 'daily_customer_precision_at_100': 0.02142857142857143}, 'logistic_regression': {'rows': 67255, 'fraud_count': 580, 'fraud_rate': 0.008623894134265111, 'average_precision': 0.4350008973626218, 'roc_auc': 0.8731684315322654, 'daily_customer_precision_at_100': 0.48000000000000004}, 'hist_gradient_boosting': {'rows': 67255, 'fraud_count': 580, 'fraud_rate': 0.008623894134265111, 'average_precision': 0.6239485132514692, 'roc_auc': 0.8849532601528257, 'daily_customer_precision_at_100': 0.5399999999999999}}, 'test_evaluated': False, 'status': 's

In [2]:
from fraud_detection_mlops.modeling.tracking import verify_model

MODEL_URI = "runs:/d3be86000fd24dc8a053dbcab778d4b6/model"
print(verify_model(MODEL_URI))

{'model_uri': 'runs:/d3be86000fd24dc8a053dbcab778d4b6/model', 'feature_count': 19, 'status': 'success'}


## Execução histórica preservada

O autor publicou e verificou os três modelos, sem refit e sem avaliar o teste.
[Recibo da execução no terminal](../../references/evidence/mlflow_execution_2026-10-07.json).
O código e a versão executada do notebook estão no
[snapshot 6ba10b5](https://github.com/wanderson42/fraud-detection-mlops/blob/6ba10b53464500a70cfba53dc696bfbca0119492/notebooks/stages/06_mlflow_tracking.ipynb).
Os comandos de migração não fazem parte do procedimento atual.

O resultado abaixo veio da célula histórica de verificação, executada pelo autor;
é um registro estático, sem simular uma nova execução das células atuais.

```json
{
  "schema_version": 1,
  "version": "mlflow_skops_v1",
  "baseline_run_id": "f6d7aca720f74316b4183f97b6d866ab",
  "baseline_manifest_sha256": "31e1ad433a5df9c42ee17b8ecfa9e8659e815d0d82b84ad39c10a483f10309e7",
  "tracking_uri": "sqlite:////home/wanderson/Documentos/fraud-detection-mlops/data/tracking/mlflow.db",
  "experiment_id": "1",
  "models": [
    {
      "model_id": "dummy_prior",
      "mlflow_run_id": "112aed5b405b4dbc90296f65eeb1fbcb",
      "model_uri": "runs:/112aed5b405b4dbc90296f65eeb1fbcb/model",
      "reused": true
    },
    {
      "model_id": "logistic_regression",
      "mlflow_run_id": "5c71bf8c647e4c878c21d057ae613103",
      "model_uri": "runs:/5c71bf8c647e4c878c21d057ae613103/model",
      "reused": true
    },
    {
      "model_id": "hist_gradient_boosting",
      "mlflow_run_id": "d3be86000fd24dc8a053dbcab778d4b6",
      "model_uri": "runs:/d3be86000fd24dc8a053dbcab778d4b6/model",
      "reused": true
    }
  ],
  "test_evaluated": false,
  "refit": false,
  "environment": {
    "mlflow": "3.17.0",
    "skops": "0.16.0",
    "scikit-learn": "1.9.1",
    "numpy": "2.5.3",
    "pandas": "3.0.6"
  },
  "export_implementation_sha256": "983a1b7c1cfa2455aab6476422cb0f93293c45f209e9f8e4d61bd10e28b759ab",
  "export_poetry_lock_sha256": "0ed7fb3173e424aa2b399313a4e9712933e4694d89774239f3caae9430d40582",
  "selected_model": "hist_gradient_boosting",
  "status": "success",
  "recorded_at_utc": "2026-10-08T00:09:59.882760+00:00"
}
```


### Runs e métricas observadas no MLflow

Output de consulta real preservado da segunda célula histórica:

```text
              model_id                           run_id   status  validation_rows  validation_fraud_count  validation_fraud_rate  validation_average_precision  validation_roc_auc  validation_daily_customer_precision_at_100  fit_seconds  predict_seconds
           dummy_prior 112aed5b405b4dbc90296f65eeb1fbcb FINISHED          67255.0                   580.0               0.008624                      0.008624            0.500000                                    0.021429     0.034777         0.000788
   logistic_regression 5c71bf8c647e4c878c21d057ae613103 FINISHED          67255.0                   580.0               0.008624                      0.435001            0.873168                                    0.480000     0.343365         0.005431
hist_gradient_boosting d3be86000fd24dc8a053dbcab778d4b6 FINISHED          67255.0                   580.0               0.008624                      0.623949            0.884953                                    0.540000     0.602763         0.036915
```

`reused: true` no registro anterior indica a consulta das runs existentes na
verificação; não demonstra uma segunda publicação idempotente.
Os tempos acima são observações daquela execução, não um benchmark de produção.


## Limites operacionais

Uma run de candidato `FINISHED` não comprova a publicação do baseline completo;
confira o manifesto e a auditoria locais. Falhas mantêm as runs individuais já
concluídas, com a falha registrada na auditoria; não há promoção automática.
Preserve SQLite e artefatos de `data/tracking` juntos. A UI permanece na porta 5001.
A preparação do fluxo nativo usa fixtures sintéticas e não altera os resultados reais
registrados acima. Hipóteses estatísticas, tuning e quality gates têm escopo próprio.
